# Notebook 5: OpenCV Introduction — Seeing With the Robot

This is the first **vision** notebook. Every previous notebook (1-4) gave
the robot ways to act on the world (LEDs, a button, motors, an ultrasonic
sensor). This one gives it a way to *see* it — a camera feed, read as
plain numeric data, and processed with OpenCV step by step: image basics,
channel order (BGR vs RGB), grayscale, resizing, a full
blur → threshold → edges → morphology progression, contours, and finally
real color detection using this project's own `src/vision/camera.py` and
`src/vision/color_detection.py` modules — ending with a small callback to
Notebook 2's RGB LED to prove vision and hardware can be composed
together.

**If you're here because the camera seems to be producing very dark
images**: skip the theory for now and go straight to the next section.
It's placed first, before any of the "image as numbers" material, so you
can check the live camera feed with the fewest possible cells run.

This notebook only drives one already-tested piece of real hardware (the
RGB LED, briefly, in Section 11) — no motors, no new safety warnings
beyond what Notebook 2 already covered.


## 1. Confirm the environment: `import cv2`

### Explanation

The whole notebook depends on OpenCV. Before anything else, confirm it
imports cleanly and check the installed version.


In [ ]:
import cv2

print(f"OpenCV version: {cv2.__version__}")


### Expected result

`OpenCV version: 5.0.0` (or similar) printed, no error.

This project installed `opencv-python-headless` on the Pi. "Headless"
means the GUI backend (Qt/GTK) that functions like `cv2.imshow()` need is
not compiled in — calling `cv2.imshow()` here would fail outright, and
even a full desktop OpenCV build's `cv2.imshow()` wouldn't work over a
remote Jupyter session anyway (there's no local window for it to open).
That's exactly why every image in this notebook is displayed by embedding
it directly in the notebook output instead, starting in the next section.

### Physical result

None — this cell only imports a library.


## 2. Camera capture and live display — do this first

This is the most important section in this notebook *right now*: it lets
you see exactly what the camera sees, directly inside Jupyter, with no
file transfer back and forth over SSH. Everything below this section
builds on `import cv2` from Section 1, not on any of the "image as
numbers" theory that follows — run just the next few cells and you'll
have a live look at the camera feed.

### Explanation

`src/vision/camera.py` wraps `picamera2`:

- `get_camera()` configures and starts the camera (default 640×480), with
  a 1-second warmup so auto-exposure/auto-white-balance can converge
  before the first real capture.
- `capture_frame(camera)` returns one frame as a NumPy array in OpenCV's
  native **BGR** channel order (blue, green, red — not red, green, blue).
  Getting this exactly right took real, documented investigative work —
  covered properly in Section 5 below.

Same `sys.path` trick as every previous notebook: add `../src` so
`from vision.camera import ...` (and later `from hardware.rgb_led import
...`) works from inside `notebooks/`.


In [ ]:
import sys
sys.path.insert(0, '../src')

from vision.camera import get_camera, capture_frame, cleanup as cleanup_camera

camera = get_camera()
print("Camera ready (640x480, 1s AE/AWB warmup already applied).")


### Expected result

`Camera ready (640x480, 1s AE/AWB warmup already applied).` printed, no
error, after a roughly 1-2 second pause (the warmup).

### Physical result

Nothing to see yet — this only starts the camera, it doesn't capture or
display a frame by itself.


### Explanation

Capture one frame and display it inline. `cv2.imshow()` doesn't work here
(Section 1), so this uses `IPython.display.Image`: encode the frame as a
JPEG in memory with `cv2.imencode()`, then hand those bytes straight to
`display()`. `cv2.imencode()` expects **BGR** input — exactly what
`capture_frame()` already returns — so no color conversion is needed for
this to display correctly.

(Aside: many OpenCV+Jupyter tutorials instead use
`matplotlib.pyplot.imshow()`, which also works well and additionally
shows pixel-coordinate axes. This notebook uses `IPython.display.Image`
instead because **matplotlib is not currently installed on this Pi** —
confirmed directly while writing this notebook. If you'd like axes/a
colorbar, `pip install matplotlib` first, then swap `show_bgr(frame)` for
`plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off')`.)


In [ ]:
from IPython.display import Image, display


def show_bgr(frame_bgr, title=None):
    """Display a BGR frame (OpenCV's native order) inline in Jupyter.

    Works for color (h, w, 3) and grayscale/mask (h, w) arrays alike -
    cv2.imencode() handles both. No BGR->RGB conversion here: imencode()
    already expects BGR, same as every other OpenCV function.
    """
    ok, buf = cv2.imencode('.jpg', frame_bgr)
    if not ok:
        raise RuntimeError("cv2.imencode failed to encode this frame as JPEG")
    if title:
        print(title)
    display(Image(data=buf.tobytes()))


frame = capture_frame(camera)
print(f"Captured frame: shape={frame.shape}, dtype={frame.dtype}")
show_bgr(frame, title="Live camera frame:")


### Expected result

`Captured frame: shape=(480, 640, 3), dtype=uint8` (or similar), followed
by an inline JPEG image showing whatever the camera currently sees.

### Physical result

> **If this image looks solid black or very dark, even in a well-lit
> room:** check for a lens cap or protective film still on the camera
> module, and make sure the camera is physically pointed out at the room
> rather than into an enclosure or against a surface.
>
> This is a **live, currently-open issue** on this exact Pi as of this
> writing — QA measured the captured frame's average brightness at
> roughly `[2, 2, 1]` out of `255` (essentially black) even at maximum
> exposure/gain in daylight, which points at a physical obstruction
> rather than a software bug. Re-run this cell after checking the lens.
> The rest of this notebook works the same regardless of what this cell
> shows, but the detection sections later on will only make visual sense
> once real light is actually reaching the sensor.


## 3. An image is just numbers

### Explanation

`frame` is an ordinary NumPy array — `frame.shape` is
`(height, width, channels)`, and indexing it with `frame[y, x]` returns
one pixel's values. There's nothing magic about an "image" here: it's
just a grid of numbers, same as any other array.


In [ ]:
height, width, channels = frame.shape
print(f"Height:   {height} pixels")
print(f"Width:    {width} pixels")
print(f"Channels: {channels}")
print(f"Total values in this frame: {frame.size:,} (= {height} x {width} x {channels})")
print(f"dtype: {frame.dtype} (each value is an integer 0-255)")

# Index a single pixel - note NumPy's (row, col) = (y, x) order, the
# opposite of the more familiar "x, y" way of saying it.
y, x = height // 2, width // 2
pixel = frame[y, x]
print(f"\nPixel at (x={x}, y={y}): {pixel}")


### Expected result

Shape numbers matching the resolution from Section 2 (e.g.
`Height: 480 pixels`, `Width: 640 pixels`, `Channels: 3`), a total of
`921,600` values, `dtype: uint8`, and one pixel's 3 values printed as a
small array (e.g. `[2 2 1]` right now, given the dark-frame issue above —
or normal-looking values once that's resolved).

### Physical result

None — this only reads data already captured in Section 2, nothing new
happens on the robot.


## 4. Channel order: OpenCV's BGR default

### Explanation

Those 3 numbers per pixel aren't in the "Red, Green, Blue" order most
people expect — OpenCV stores color images as **Blue, Green, Red**
(`BGR`), a historical convention from early Windows camera/codec
libraries that OpenCV has kept for backward compatibility ever since.
`capture_frame()` returns frames already in this BGR order, matching
every other OpenCV function used later in this notebook
(`cv2.cvtColor`, `cv2.imwrite`, the HSV thresholds in
`color_detection.py`, ...).


In [ ]:
b, g, r = frame[y, x]
print(f"Blue:  {b}")
print(f"Green: {g}")
print(f"Red:   {r}")


### Expected result

Three integers, matching the 3 values printed for the same pixel in
Section 3, just unpacked and labeled individually.

### Physical result

None.

Worth remembering as you go: if code ever assumes an array is RGB when
it's actually BGR (or vice versa), red and blue silently swap — a classic
bug that produces an image that still *looks* like a photo, just with the
wrong colors, so it's easy to miss. Section 5 shows this for real.


## 5. Converting to RGB, and why it actually matters

### Explanation

`cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)` reorders the channels to
Red-Green-Blue. Most tools *outside* OpenCV — matplotlib, PIL, most
machine-learning image pipelines, a browser's `<img>` tag — expect RGB,
not BGR. Handing a BGR array to one of those without converting first
doesn't raise an error; it just quietly swaps red and blue in whatever
gets displayed or fed to a model.

This isn't a hypothetical concern for this project — it's exactly the
investigation `src/vision/camera.py`'s docstring documents in detail.
picamera2's default capture format for this sensor is named `"XBGR8888"`,
but reading picamera2's own source and comparing captured-array channel
means against a same-scene JPEG (whose channel order is unambiguous)
showed the array's actual in-memory layout is **the reverse of what the
format name suggests** — a real, documented picamera2/libcamera
naming-vs-layout quirk, not something safe to assume. `capture_frame()`
drops the unused 4th padding channel and does an explicit
`cv2.cvtColor(..., cv2.COLOR_RGB2BGR)` specifically so this conversion is
visible in the code, not silently relied upon. Worth reading that
module's full docstring at least once.


In [ ]:
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

print("Correct: a BGR frame passed to a BGR-expecting JPEG encoder")
show_bgr(frame)

print("Deliberately WRONG: an RGB-ordered array passed to that same BGR-expecting encoder")
show_bgr(frame_rgb)


### Expected result

Two images. If the frame has real color content in it, the second one
will visibly have red and blue swapped compared to the first (e.g. a blue
sky would look orange-ish, warm skin tones would look blue-ish). If the
frame is still very dark per Section 2's open issue, both may look
nearly identical (black either way) — re-run this cell once that's
resolved to see the swap clearly.

### Physical result

None — screen-only.


## 6. Grayscale

### Explanation

`cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)` collapses the 3 color channels
into 1 brightness channel. Grayscale matters for real, practical
reasons: it's a third of the data to move and process, and a large
fraction of classic CV algorithms — blurring, thresholding, edge
detection, contours (all coming up next) — only care about brightness
patterns, not color, so there's no accuracy lost by dropping color for
those steps.


In [ ]:
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
print(f"Color shape:     {frame.shape}")
print(f"Grayscale shape: {gray.shape}  (no channel dimension - just height x width)")
show_bgr(gray, title="Grayscale:")


### Expected result

`Grayscale shape: (480, 640)` (2 dimensions, not 3), followed by a
grayscale version of the same frame.

### Physical result

None.


## 7. Resizing

### Explanation

`cv2.resize(frame, (new_width, new_height))` changes an image's
dimensions. On an embedded board like this Pi, resizing down matters for
two concrete reasons: less data means faster per-frame processing (real
-time CV needs to keep up with the camera's frame rate, not just
eventually finish), and later machine-learning models (Notebooks 06-07)
expect a **fixed** input resolution regardless of what the camera
natively captures.


In [ ]:
small = cv2.resize(frame, (320, 240))
print(f"Original: {frame.shape}")
print(f"Resized:  {small.shape}")
show_bgr(small, title="Resized to 320x240:")


### Expected result

`Original: (480, 640, 3)` then `Resized: (240, 320, 3)`, followed by the
same scene at a quarter the pixel count — same content, visibly blockier
if you look closely.

### Physical result

None.


## 8. From pixels to structure: a processing progression

### Explanation

A classic CV pipeline, one small controllable step at a time, each
building on the last: **blur** (smooth out small-scale sensor noise) →
**threshold** (turn brightness into a clean black/white decision) →
**edges** (find sharp brightness transitions) → **morphology** (clean up
the result — close small gaps, thicken thin lines). Each step below is
displayed separately so you can see exactly what it changes.


In [ ]:
blur = cv2.GaussianBlur(gray, (5, 5), 0)
show_bgr(blur, title="Blurred (GaussianBlur, 5x5 kernel):")


### Expected result

A visibly softer/smoother version of the grayscale image — fine-grained
noise smeared out, larger shapes still recognizable. Blurring first means
the later threshold/edge steps react to real structure in the scene
rather than to individual noisy pixels.

### Physical result

None.


In [ ]:
_, thresh = cv2.threshold(blur, 60, 255, cv2.THRESH_BINARY)
show_bgr(thresh, title="Thresholded (brightness > 60 -> white, else black):")


### Expected result

A pure black-and-white image: pixels brighter than 60 become solid white,
everything else solid black.

**Given the dark-frame issue from Section 2, this will likely come back
almost entirely black right now** — nearly every pixel is below the `60`
cutoff. That's the same underlying camera issue showing up here, not a
bug in this cell. Try lowering the threshold (e.g. `10`) to see some
structure in a dark frame, or better, resolve the lighting issue first
and re-run.

### Physical result

None.


In [ ]:
edges = cv2.Canny(blur, 50, 150)
show_bgr(edges, title="Edges (Canny):")


### Expected result

Thin white outlines wherever brightness changes sharply, black
everywhere else. Likely sparse or empty right now, for the same reason as
the threshold step above.

### Physical result

None.


In [ ]:
dilated = cv2.dilate(edges, None, iterations=1)
show_bgr(dilated, title="Dilated edges (thicker lines, small gaps closed):")


### Expected result

The same edge pattern as before, just thicker and with small gaps
between nearby edge pixels closed up. `cv2.dilate`/`cv2.erode` (or
`cv2.morphologyEx` for combinations of the two) are standard cleanup
steps applied to thresholded or edge images before extracting contours,
which is exactly what's next.

### Physical result

None.


## 9. Finding boundaries: contours

### Explanation

`cv2.findContours()` finds the boundaries of connected white regions in a
binary (black/white) image like `thresh` from Section 8, returning a list
of point arrays — one per boundary found. `cv2.boundingRect()` turns one
of those into a simple axis-aligned box. Drawn on a **copy of the
original color frame** (not the binary mask itself) so the result is easy
to look at.


In [ ]:
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print(f"Found {len(contours)} contour(s).")

annotated = frame.copy()

if contours:
    largest = max(contours, key=cv2.contourArea)
    bx, by, bw, bh = cv2.boundingRect(largest)
    cx, cy = bx + bw // 2, by + bh // 2

    cv2.drawContours(annotated, [largest], -1, (0, 255, 0), 2)          # green outline
    cv2.rectangle(annotated, (bx, by), (bx + bw, by + bh), (255, 0, 0), 2)  # blue box
    cv2.circle(annotated, (cx, cy), 5, (0, 0, 255), -1)                 # red center dot

    print(f"Largest contour: area={cv2.contourArea(largest):.0f}px, "
          f"bbox=({bx},{by},{bw},{bh}), center=({cx},{cy})")
else:
    print("No contours found - with a very dark/thresholded frame this is "
          "expected, nothing crosses the brightness cutoff.")

show_bgr(annotated, title="Annotated frame:")


### Expected result

**If contours were found**: a count greater than 0, the largest one's
area/bbox/center printed, and the displayed frame shows a green outline,
a blue bounding box, and a red center dot around the brightest region.

**If none were found** (likely right now, per Section 8): the
"No contours found..." message, and the displayed image is just the
plain frame with no annotation — expected given the ongoing dark-frame
issue, not a bug in this cell.

### Physical result

None.


## 10. Color detection with `color_detection.py`

### Explanation

Everything so far worked on brightness alone. Real color detection uses
**HSV** instead of BGR/RGB: Hue (the actual color, 0-179 in OpenCV),
Saturation (how vivid vs. washed-out), and Value (brightness). Because
lighting changes shift all of BGR's channels together but mostly leave
Hue alone, thresholding on Hue is far more robust to normal lighting
variation than thresholding on raw BGR/RGB ever could be.

`src/vision/color_detection.py` implements this already:

- `detect_color(frame, color_name)` → a binary mask for `"RED"`,
  `"GREEN"`, or `"YELLOW"` (RED uses two HSV ranges OR'd together, to
  handle Hue being circular — 0 and 179 are both "red" — see that
  module's docstring for the full gotcha).
- `largest_blob(mask, min_area=200)` → the biggest matching region's
  centroid, bounding box, and area, or `None` if nothing big enough was
  found.

**Be honest about where these numbers come from**: QA verified the
*algorithm* is correct using synthetic test images (including the red
hue-wraparound edge cases), but the specific HSV threshold values in
`HSV_RANGES` are standard/reference starting points, **not yet validated
against a real red/green/yellow object on this hardware** — blocked by
the same camera issue flagged in Section 2. Once that's resolved, hold up
real colored objects (construction paper, LEGO bricks, whatever's on
hand) and confirm each color is actually detected; if one isn't, the
module's docstring recommends widening the Saturation/Value bounds first
(lighting/washout issues) before touching the Hue bounds.


In [ ]:
from vision.color_detection import detect_color, largest_blob, HSV_RANGES

frame = capture_frame(camera)  # fresh frame

for color_name in HSV_RANGES:
    mask = detect_color(frame, color_name)
    blob = largest_blob(mask)
    annotated = frame.copy()

    if blob is not None:
        bx, by, bw, bh = blob["bbox"]
        cx, cy = blob["centroid"]
        cv2.rectangle(annotated, (bx, by), (bx + bw, by + bh), (0, 255, 255), 2)
        cv2.circle(annotated, (cx, cy), 5, (255, 255, 255), -1)
        print(f"{color_name}: detected - {blob}")
    else:
        print(f"{color_name}: not detected (no blob >= 200px)")

    show_bgr(annotated, title=f"{color_name} detection:")


### Expected result

Three rounds (RED, GREEN, YELLOW), each printing either
`detected - {...}` (with the box/centroid drawn in the displayed image)
or `not detected`.

**Given the current camera darkness and not-yet-validated thresholds,
expect "not detected" for all three right now** — that's expected, not a
bug. Once the lighting issue is resolved, hold up real red/green/yellow
objects in front of the camera and re-run this cell.

### Physical result

None — vision only, no hardware driven yet.


## 11. Putting it together: detected color → RGB LED

> ## Note — real hardware, low risk
> This section drives the RGB LED from Notebook 2 — already tested, low
> current, no motors involved. Nothing else moves.

### Explanation

`color_detection.py` is vision-only by design (no GPIO, no actuator
imports) and `rgb_led.py` is hardware-only — composing "detected color →
matching LED color" happens **here, at the notebook level**, the same
principle `src/robot/obstacle_avoidance.py` used to compose `motor.py`
and `ultrasonic.py` without either module reaching into the other.

Check RED, GREEN, and YELLOW; light the LED to match whichever has the
largest detected blob; turn it off if nothing was detected.


In [ ]:
from hardware.rgb_led import get_rgb_led, set_red, set_green, set_yellow, set_off, cleanup as cleanup_rgb

rgb = get_rgb_led()
frame = capture_frame(camera)

color_to_led = {"RED": set_red, "GREEN": set_green, "YELLOW": set_yellow}

detected_color = None
best_area = 0
for color_name, set_fn in color_to_led.items():
    blob = largest_blob(detect_color(frame, color_name))
    if blob is not None and blob["area"] > best_area:
        detected_color = color_name
        best_area = blob["area"]

if detected_color is not None:
    color_to_led[detected_color](rgb)
    print(f"Detected {detected_color} (area={best_area:.0f}px) - RGB LED set to {detected_color}.")
else:
    set_off(rgb)
    print("No color detected - RGB LED off.")


### Expected result

A line stating which color won and that the LED was set to match, or
`No color detected - RGB LED off.`.

**Given Section 10's current results, expect "No color detected" right
now.** Once the camera issue is fixed and a real colored object is in
frame, re-run this cell and the LED should physically light up matching
whichever color was detected.

### Physical result

**Right now**: LED off (or unchanged, if it was already off).

**Once working**: the RGB LED lights up red, green, or yellow to match
whatever colored object the camera sees — the first time in this project
vision output has driven a physical actuator.


## 12. Where this is headed: object detection (concept only)

Everything above — contours, HSV color detection — finds "a blob of a
particular brightness/color". It has no idea *what* that blob actually
is; a red ball and a red sock would be detected identically. **Object
detection** is a different, more general approach: a trained model looks
at an image and outputs, for each object it recognizes:

- a **class label** (what it thinks the object is — e.g. "person",
  "car", "bottle" — drawn from whatever set of classes the model was
  trained on),
- a **bounding box** (where it is in the frame), and
- a **confidence score** (how sure the model is).

Conceptually: `image → model → [(class, bbox, confidence), ...]`. Unlike
color detection, this doesn't require the object to be a specific known
color — it generalizes to shape/texture/context learned from training
data.

This notebook doesn't run a model — that's genuinely new territory,
covered starting in **Notebooks 06-07**, which introduce an SSD
(Single Shot Detector) model running on this same Pi. Everything built in
this notebook (camera capture, BGR/RGB handling, resizing to a fixed
input size) is exactly the preprocessing those notebooks will reuse.


## Cleanup

### Explanation

Release the camera and the RGB LED's GPIO pins, same "close what you
opened" pattern as every previous notebook.


In [ ]:
cleanup_camera(camera)
cleanup_rgb(rgb)
print("Camera and RGB LED released.")


### Expected result

`Camera and RGB LED released.` printed, no error.

### Physical result

LED off, camera stopped.


## Recap

- **The camera-capture-and-display section (Section 2) was deliberately
  placed first**, ahead of any theory, specifically so the camera feed
  could be checked visually in Jupyter as soon as possible — which is the
  whole reason this notebook exists right now. If Section 2 showed a
  black/very-dark image, that's a known, currently-open issue (mean BGR
  ≈ `[2, 2, 1]`/255 even at max exposure/gain in daylight) — check for a
  lens cap/protective film and that the camera is pointed at the room.
- An image is just a NumPy array: `frame.shape` is
  `(height, width, channels)`, and OpenCV's default channel order is
  **BGR**, not RGB — a real, hardware-verified investigation in
  `src/vision/camera.py`'s docstring exists specifically because getting
  this wrong silently swaps red and blue.
- Grayscale, resizing, blur → threshold → edges → morphology, and
  contours are the classic small, composable building blocks most
  "simple" CV (as opposed to ML-based) tasks are built from.
- **HSV**, not BGR/RGB, is what `color_detection.py` thresholds on,
  because Hue stays comparatively stable under lighting changes that
  shift BGR/RGB's channels together. RED needs two HSV ranges OR'd
  together because Hue wraps around at the 0/179 seam.
- `color_detection.py`'s detection *algorithm* is verified (synthetic
  tests, including the red-wraparound edge case); its specific HSV
  threshold **values** are reference defaults, not yet confirmed against
  real objects on this hardware — that's still open, blocked by the same
  camera issue.
- Vision and hardware modules stay separate (`color_detection.py` never
  imports `rgb_led.py`) — composing "detected color → LED color" happens
  at the notebook level, the same pattern `obstacle_avoidance.py` used for
  `motor.py` + `ultrasonic.py`.
- Real object detection (class + bbox + confidence from a trained model,
  not just color/brightness blobs) is next, in Notebooks 06-07.


## Exercises

**1. Detect a different/additional color.**
`color_detection.py` only defines RED, GREEN, YELLOW in `HSV_RANGES`, but
`detect_color()` just looks that dict up by name — you can extend it at
the notebook level without touching the module:

```python
from vision.color_detection import HSV_RANGES
HSV_RANGES["BLUE"] = [((90, 80, 40), (130, 255, 255))]  # a reasonable starting guess

mask = detect_color(frame, "BLUE")
blob = largest_blob(mask)
print(blob)
```

Try it with a real blue object once the camera issue is resolved, and
adjust the H/S/V bounds if it isn't detected well.

**2. Change HSV thresholds and observe the effect.**
Pick one color (e.g. `"YELLOW"`) and temporarily widen or narrow its
Saturation/Value bounds directly in `HSV_RANGES["YELLOW"]` (in this
notebook, not in `color_detection.py`), then re-run Section 10's
detection cell. Does a wider S/V range pick up more of a washed-out or
shadowed object? Does a narrower one lose it?

**3. Draw the center point with a label.**
Section 9 and 10 already draw a center dot — extend either one with
`cv2.putText()` to also print the pixel coordinates directly on the
image next to the dot, e.g.
`cv2.putText(annotated, f"({cx},{cy})", (cx + 10, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)`.

**4. Count multiple detected objects, not just the largest.**
`largest_blob()` only ever returns the single biggest contour. Write your
own small function in this notebook (not editing `color_detection.py`)
that mirrors its logic but keeps every contour at or above
`min_area`, not just the largest:

```python
def all_blobs(mask, min_area=200):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    blobs = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < min_area:
            continue
        x, y, w, h = cv2.boundingRect(c)
        m = cv2.moments(c)
        cx = int(m["m10"] / m["m00"]) if m["m00"] else x + w // 2
        cy = int(m["m01"] / m["m00"]) if m["m00"] else y + h // 2
        blobs.append({"centroid": (cx, cy), "bbox": (x, y, w, h), "area": area})
    return blobs
```

Use it to draw a box around *every* detected blob of a color (e.g.
several red objects in frame at once) and print how many were found.
